In [1]:
# CELL 1 — Imports, locked config, load Phase 1 output (Pavia University)
import numpy as np
import torch
import torch.nn as nn
import math
import time
import json
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

N_QUBITS = 4
ENTANGLING_LAYERS = 2
D_MODEL = 64
D_FF = 128
N_HEADS = 4
N_TOKENS = 225
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
EPOCHS = 50
LR = 2e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SEEDS = [42, 43, 44]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

data = np.load("preprocessed/PaviaUniversity.npz")
train_tokens = torch.tensor(data["train_tokens"], dtype=torch.float32)
train_labels = torch.tensor(data["train_labels"] - 1, dtype=torch.long)
val_tokens = torch.tensor(data["val_tokens"], dtype=torch.float32)
val_labels = torch.tensor(data["val_labels"] - 1, dtype=torch.long)
test_tokens = torch.tensor(data["test_tokens"], dtype=torch.float32)
test_labels = torch.tensor(data["test_labels"] - 1, dtype=torch.long)

K_DIM = train_tokens.shape[-1]
N_CLASSES = int(train_labels.max().item()) + 1
print(f"Pavia University: k={K_DIM}, classes={N_CLASSES}")
print(f"train={train_tokens.shape}, val={val_tokens.shape}, test={test_tokens.shape}")

Using device: cuda
Pavia University: k=10, classes=9
train=torch.Size([4277, 225, 10]), val=torch.Size([4277, 225, 10]), test=torch.Size([34222, 225, 10])


In [2]:
# CELL 2 — QuantFormer model (dropout=0.0 locked in per Phase 2 findings)
import pennylane as qml

QUANTUM_DEVICE_NAME = "default.qubit"
DIFF_METHOD = "backprop"

def build_quantum_layer():
    dev = qml.device(QUANTUM_DEVICE_NAME, wires=N_QUBITS)
    weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=ENTANGLING_LAYERS, n_wires=N_QUBITS)

    @qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]

    return qml.qnn.TorchLayer(circuit, {"weights": weight_shape})


class QuantumTokenEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.angle_proj = nn.Linear(k_dim, N_QUBITS)
        self.q_layer = build_quantum_layer()
        self.out_proj = nn.Linear(N_QUBITS, D_MODEL)

    def forward(self, tokens):
        b, n, k = tokens.shape
        theta = math.pi * torch.tanh(self.angle_proj(tokens))
        flat = theta.reshape(b * n, N_QUBITS)
        q_out = self.q_layer(flat).reshape(b, n, N_QUBITS)
        return self.out_proj(q_out)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_tokens, d_model):
        super().__init__()
        pe = torch.zeros(n_tokens, d_model)
        pos = torch.arange(0, n_tokens, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe


class QuantFormer(nn.Module):
    def __init__(self, k_dim, n_classes):
        super().__init__()
        self.q_encoder = QuantumTokenEncoder(k_dim)
        self.pos_enc = SinusoidalPositionalEncoding(N_TOKENS, D_MODEL)
        self.encoder = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True,
            dropout=0.0,  # LOCKED per Phase 2 diagnostic — was an unintended default
        )
        self.classifier = nn.Linear(D_MODEL, n_classes)

    def forward(self, tokens):
        x = self.q_encoder(tokens)
        x = self.pos_enc(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.classifier(x)

print("Cell 2 loaded: QuantFormer (dropout locked to 0.0) ready.")

Cell 2 loaded: QuantFormer (dropout locked to 0.0) ready.


In [3]:
# CELL 3 — Training loop with best-checkpoint selection (locked per Phase 2)

def get_param_groups(model):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if "q_layer" in name:
            no_decay.append(param)
        else:
            decay.append(param)
    return [
        {"params": decay, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay, "weight_decay": 0.0},
    ]

def train_one_seed(seed, k_dim, n_classes, train_tokens, train_labels, val_tokens, val_labels):
    torch.manual_seed(seed)
    model = QuantFormer(k_dim, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state = -1, None

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        epoch_loss = 0.0
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            epoch_loss += loss.item() * len(idx)

        model.eval()
        with torch.no_grad():
            val_out = model(val_tokens.to(DEVICE))
            val_loss = criterion(val_out, val_labels.to(DEVICE)).item()
            val_acc = accuracy_score(val_labels, val_out.argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(epoch_loss / n_train)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  seed={seed} epoch={epoch+1}/{EPOCHS} val_acc={val_acc:.4f} (best={best_val_acc:.4f})")

    model.load_state_dict(best_state)
    return model, history

print("Cell 3 loaded: train_one_seed() with best-checkpoint selection ready.")

Cell 3 loaded: train_one_seed() with best-checkpoint selection ready.


In [4]:
# CELL 4 — Evaluation
def evaluate(model, test_tokens, test_labels, n_classes):
    model.eval()
    with torch.no_grad():
        out = model(test_tokens.to(DEVICE))
        preds = out.argmax(dim=1).cpu().numpy()
    labels_np = test_labels.numpy()

    oa = accuracy_score(labels_np, preds)
    cm = confusion_matrix(labels_np, preds, labels=list(range(n_classes)))
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    aa = per_class_acc.mean()
    kappa = cohen_kappa_score(labels_np, preds)

    return {"OA": oa, "AA": aa, "kappa": kappa, "confusion_matrix": cm.tolist(),
            "per_class_accuracy": per_class_acc.tolist()}

print("Cell 4 loaded: evaluate() ready.")

Cell 4 loaded: evaluate() ready.


In [5]:
# CELL 5 — Pavia University reproduction, 3 seeds, checkpoint against paper's reported numbers
PAPER_OA, PAPER_AA, PAPER_KAPPA = 0.9994, None, None  # paper reports OA=99.94%; AA/kappa not separately confirmed here

results = []
for seed in SEEDS:
    print(f"\n=== Pavia University, seed={seed} ===")
    start = time.time()
    model, history = train_one_seed(seed, K_DIM, N_CLASSES,
                                     train_tokens, train_labels, val_tokens, val_labels)
    elapsed = time.time() - start
    metrics = evaluate(model, test_tokens, test_labels, N_CLASSES)
    metrics["seed"] = seed
    metrics["train_time_sec"] = elapsed
    results.append(metrics)
    print(f"  OA={metrics['OA']:.4f}  AA={metrics['AA']:.4f}  kappa={metrics['kappa']:.4f}  "
          f"(train time: {elapsed/60:.1f} min)")

oa_vals = [r["OA"] for r in results]
aa_vals = [r["AA"] for r in results]
kappa_vals = [r["kappa"] for r in results]

print(f"\n{'='*60}\nPAVIA UNIVERSITY — 3-SEED SUMMARY (mean ± std)\n{'='*60}")
print(f"OA:    {np.mean(oa_vals):.4f} ± {np.std(oa_vals):.4f}   (paper: {PAPER_OA})")
print(f"AA:    {np.mean(aa_vals):.4f} ± {np.std(aa_vals):.4f}")
print(f"kappa: {np.mean(kappa_vals):.4f} ± {np.std(kappa_vals):.4f}")

within_1pct = abs(np.mean(oa_vals) - PAPER_OA) <= 0.01
print(f"\nWithin 1% OA of paper: {'YES' if within_1pct else 'NO'} "
      f"(diff = {abs(np.mean(oa_vals) - PAPER_OA)*100:.2f} percentage points)")

with open("phase3_pavia_results.json", "w") as f:
    json.dump({"per_seed": results, "summary": {
        "OA_mean": float(np.mean(oa_vals)), "OA_std": float(np.std(oa_vals)),
        "AA_mean": float(np.mean(aa_vals)), "AA_std": float(np.std(aa_vals)),
        "kappa_mean": float(np.mean(kappa_vals)), "kappa_std": float(np.std(kappa_vals)),
        "within_1pct_OA": bool(within_1pct),
    }}, f, indent=2)
print("\nSaved phase3_pavia_results.json")


=== Pavia University, seed=42 ===
  seed=42 epoch=1/50 val_acc=0.7477 (best=0.7477)
  seed=42 epoch=10/50 val_acc=0.9652 (best=0.9652)
  seed=42 epoch=20/50 val_acc=0.9581 (best=0.9771)
  seed=42 epoch=30/50 val_acc=0.9801 (best=0.9895)
  seed=42 epoch=40/50 val_acc=0.9867 (best=0.9895)
  seed=42 epoch=50/50 val_acc=0.9878 (best=0.9895)
  OA=0.9845  AA=0.9792  kappa=0.9795  (train time: 7.1 min)

=== Pavia University, seed=43 ===
  seed=43 epoch=1/50 val_acc=0.7961 (best=0.7961)
  seed=43 epoch=10/50 val_acc=0.9698 (best=0.9698)
  seed=43 epoch=20/50 val_acc=0.9776 (best=0.9815)
  seed=43 epoch=30/50 val_acc=0.9822 (best=0.9904)
  seed=43 epoch=40/50 val_acc=0.9878 (best=0.9930)
  seed=43 epoch=50/50 val_acc=0.9935 (best=0.9942)
  OA=0.9949  AA=0.9905  kappa=0.9932  (train time: 8.7 min)

=== Pavia University, seed=44 ===
  seed=44 epoch=1/50 val_acc=0.7159 (best=0.7159)
  seed=44 epoch=10/50 val_acc=0.9553 (best=0.9581)
  seed=44 epoch=20/50 val_acc=0.9736 (best=0.9822)
  seed=44 epo